# 03. Tokenizer Anatomy — Part 1: 文字トークナイザ

## 学習目標

- テキスト ↔ token ID 列の対応を**1文字単位で完全に**追えるようになる
- `decode(encode(x)) == x`（round-trip）の意味と、それが破れる唯一の条件（未知文字）を知る
- 文字トークナイザの限界（語彙サイズ・系列長・被覆率）を実測し、BPE（M2）の動機を得る

## 数式

語彙 $\mathcal{V} = \{\text{特殊トークン6種}\} \cup \{\text{学習テキストの全文字}\}$ に対し

$$\operatorname{encode}: \text{str} \to \mathbb{N}^*, \quad \operatorname{decode}: \mathbb{N}^* \to \text{str}$$

$$\operatorname{decode}(\operatorname{encode}(x)) = x \quad (\forall x: \text{全文字が } \mathcal{V} \text{ 内})$$

特殊トークンは固定ID: `<PAD>`=0, `<UNK>`=1, `<BOS>`=2, `<EOS>`=3, `<USER>`=4, `<ASSISTANT>`=5

In [1]:
# 共通セットアップ（全ノートブック同一）
import warnings
warnings.filterwarnings("ignore")

import math
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"

from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT={ROOT}")
print(f"device={DEVICE}, torch={torch.__version__}")

ROOT=/home/kazumasa/projects/jp_llm_lab
device=cuda, torch=2.11.0+cu128


In [2]:
from jp_llm_lab.data.sample_corpus import load_sample_corpus
from jp_llm_lab.tokenization.char_tokenizer import CharTokenizer

corpus = load_sample_corpus("kokoro")  # 夏目漱石『こころ』（青空文庫・パブリックドメイン）
tok = CharTokenizer.train([corpus])
print(f"コーパス: {len(corpus):,} 文字")
print(f"語彙サイズ: {tok.vocab_size:,}（特殊6 + 文字 {tok.vocab_size - 6:,} 種）")
print(f"特殊トークン: {tok.itos[:6]}")
print(f"文字部分の先頭20: {tok.itos[6:26]}")

コーパス: 162,481 文字
語彙サイズ: 2,068（特殊6 + 文字 2,062 種）
特殊トークン: ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<USER>', '<ASSISTANT>']
文字部分の先頭20: ['\n', '―', '…', '※', '\u3000', '、', '。', '々', '「', '」', '『', '』', 'あ', 'い', 'う', 'え', 'お', 'か', 'が', 'き']


In [3]:
# 手計算例: 1文字ずつ表引きするだけ
s = "私は猫である。"
ids = tok.encode(s)
pd.DataFrame({"char": list(s), "token_id": ids})

,char,token_id
0,私,1382
1,は,59
2,猫,1262
3,で,51
4,あ,18
5,る,84
6,。,12


In [4]:
# round-trip 恒等式（テストでも保証済み）と、破れる唯一のケース
assert tok.decode(tok.encode(corpus)) == corpus
print("decode(encode(corpus)) == corpus ✓")

print("未知文字の例:", tok.encode("𠮷"), "→", tok.decode(tok.encode("𠮷")))  # <UNK>=1

decode(encode(corpus)) == corpus ✓
未知文字の例: [1] → <UNK>


In [5]:
# 頻出文字 top30
freq = Counter(corpus)
top = freq.most_common(30)
labels = [c.replace("\n", "⏎").replace("　", "␣") for c, _ in top]
fig = go.Figure(go.Bar(x=labels, y=[n for _, n in top], marker_color="#1f77b4"))
fig.update_layout(title="『こころ』頻出文字 top30", xaxis_title="文字", yaxis_title="出現回数",
                  template="plotly_white", height=380)
fig.show()

In [6]:
# 文字種構成（ひらがな/カタカナ/漢字/英数字/記号）
def char_class(ch):
    o = ord(ch)
    if 0x3040 <= o <= 0x309F:
        return "ひらがな"
    if 0x30A0 <= o <= 0x30FF:
        return "カタカナ"
    if 0x4E00 <= o <= 0x9FFF:
        return "漢字"
    if ch.isascii() and ch.isalnum():
        return "英数字"
    return "記号・空白・その他"

occ = Counter(char_class(c) for c in corpus)                # 出現数ベース
uniq = Counter(char_class(c) for c in set(corpus))          # 語彙ベース
df = pd.DataFrame({"出現数": occ, "語彙数（異なり）": uniq}).fillna(0).astype(int)
fig = go.Figure()
for col in df.columns:
    fig.add_trace(go.Bar(x=df.index, y=df[col] / df[col].sum(), name=col))
fig.update_layout(barmode="group", title="文字種構成: 出現ベース vs 語彙ベース",
                  yaxis_title="割合", template="plotly_white", height=380)
fig.show()
df

,出現数,語彙数（異なり）
漢字,44443,1925
記号・空白・その他,12636,16
ひらがな,105219,72
カタカナ,183,49


**How to read**: 出現ベース（テキストの何%か）と語彙ベース（異なり文字の何%か）は大きく食い違う。
ひらがなは**出現**の大半を占めるが種類は~80。漢字は逆に**語彙**の大半を占める。
これが「文字トークナイザの語彙サイズは漢字の異なり数で決まる」という構造。

In [7]:
# 被覆率（OOV）: 『こころ』で学習した語彙で『走れメロス』を符号化すると？
merosu = load_sample_corpus("hashire_merosu")
ids_m = tok.encode(merosu)
unk_rate = sum(1 for i in ids_m if i == 1) / len(ids_m)
unk_chars = sorted({c for c in set(merosu) if c not in tok.stoi})
print(f"『走れメロス』{len(merosu):,} 文字中 UNK率 = {unk_rate:.2%}")
print(f"未知文字 {len(unk_chars)} 種（例）: {unk_chars[:30]}")

『走れメロス』9,915 文字中 UNK率 = 2.82%
未知文字 96 種（例）: ['ィ', 'ウ', 'セ', 'ゼ', 'ヌ', '亭', '佞', '佳', '刑', '匹', '叟', '叶', '吉', '后', '吏', '呆', '呟', '哉', '嗄', '噴', '囁', '壺', '奸', '婿', '宝', '宴', '巡', '巣', '律', '怺']


## Observation / Interpretation / Caveat

- **Observation**: 語彙 ~2,000 のうち漢字が大半。別コーパス（メロス）では未知文字が発生し、
  UNK は**復元不可能な情報損失**になる（round-trip が破れる）。
- **Interpretation**: 文字トークナイザは (1) 実装が単純で対応が完全に追える、
  (2) 系列長=文字数で長い、(3) 未知**文字**に弱い。サブワード（BPE, M2）は
  バイト/文字の結合で語彙を「よく出る並び」に割り当て、この3点のトレードオフを変える。
- **Caveat**: UNK率はコーパスの組に依存する。「日本語一般」の値ではない。

## 確認問題

1. `<UNK>` が存在するのに round-trip テストが成立するのはなぜか（どんな入力に対する保証か）。
2. 語彙を『こころ』+『メロス』の和集合にすると何が改善し、何が解決しないか。
3. 文字トークナイザで context 256 は「何文字」か。BPE（1トークン≈2-3文字）では何文字相当か。

**次へ**: [04_bigram_language_model](04_bigram_language_model.ipynb)